In [0]:
%pip install langchain langchain-community chromadb sentence-transformers langchain-text-splitters beautifulsoup4 lxml

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ba207d32-155f-4115-860a-3afe8a5eede6/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten
/home/spark-ba207d32-155f-4115-860a-3a/.ipykernel/65/command-6468244426884792-4009195345:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [0]:
# 1. live ingestion: pulling a massive real world document from the web
target_url = "https://en.wikipedia.org/wiki/Wakeboarding"
print(f"Scraping live data from {target_url}...")
loader = WebBaseLoader(target_url)
web_documents = loader.load()

Scraping live data from https://en.wikipedia.org/wiki/Wakeboarding...


In [0]:
# 2. advanced chunking: split the massive doc, but keep the metadata (source URL) attached to each chunk
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # Larger chunks for better context
    chunk_overlap=50
)
chunks = text_splitter.split_documents(web_documents)
print(f"Successfully chopped the webpage into {len(chunks)} searchable chunks!")

Successfully chopped the webpage into 52 searchable chunks!


In [0]:
# 3. vector storage: embed the chunks into a new database
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# we use from_documents instead of from_texts to preserve the metadata
advanced_vector_store = Chroma.from_documents(chunks, embeddings)
print("Enterprise Vector Database built successfully!\n")

/home/spark-ba207d32-155f-4115-860a-3a/.ipykernel/65/command-6468244426884795-3561276750:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Enterprise Vector Database built successfully!



In [0]:
# 4. advanced retrieval: query the database and return the "Relevance Score" and "Source Link"
user_question = "What are the common tricks or maneuvers?"
print(f"Captain asks: {user_question}\n")

# k=2 returns the top 2 best matching paragraphs
results = advanced_vector_store.similarity_search_with_score(user_question, k=2)

print("--- TOP RETRIEVED SOURCES ---")
for i, (doc, score) in enumerate(results):
    print(f"\nMatch {i+1} (Mathematical Distance: {score:.4f}):")
    print(f"Source URL: {doc.metadata.get('source')}")
    print(f"Content Snippet: {doc.page_content}")

Captain asks: What are the common tricks or maneuvers?

--- TOP RETRIEVED SOURCES ---

Match 1 (Mathematical Distance: 0.8996):
Source URL: https://en.wikipedia.org/wiki/Wakeboarding
Content Snippet: More advanced tricks call for what is known as inversion or “inverts”. An invert is considered any action where the board is above the head of the rider. This does not necessarily mean the rider is fully upside down, as evidenced by the Raley, a trick where the rider extends their body parallel to the water bending at the knees to achieve inversion.[24]

Match 2 (Mathematical Distance: 0.9025):
Source URL: https://en.wikipedia.org/wiki/Wakeboarding
Content Snippet: The heel side backflip, otherwise known as a tantrum, is often regarded as the first and easiest invert to learn as an intermediate level rider. This is because the shape of the wake or the “kicker” (a type of floating ramp, used primarily at cable parks, that mimics the shape of a wake) naturally initiates the sequence of motio